In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation
import mediapy

import mujoco

from motrixsim import SceneData, load_model, step

from gs_playground import ROOT_PATH
from gaussian_renderer import GSRendererMuJoCo, GSRendererMotrixSim

mjcf_path = ROOT_PATH.parent / "models" / "robots" / "manipulation" / "franka_robotiq" / "xmls" / "pick_fruit.xml"

_ASSETS_FRANKA_DIR = ROOT_PATH.parent / "models" / "robots" / "manipulation" / "franka_robotiq"
_ASSETS_BANANA_DIR = ROOT_PATH.parent / "models" / "tasks" / "table30" / "03_arrange_fruits_in_basket"
gaussians = {
    "link0" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link0.ply").as_posix(),
    "link1" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link1.ply").as_posix(),
    "link2" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link2.ply").as_posix(),
    "link3" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link3.ply").as_posix(),
    "link4" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link4.ply").as_posix(),
    "link5" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link5.ply").as_posix(),
    "link6" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link6.ply").as_posix(),
    "link7" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link7.ply").as_posix(),

    "robotiq_base"      : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "robotiq_base.ply").as_posix(),
    "left_driver"       : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_driver.ply").as_posix(),
    "left_coupler"      : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_coupler.ply").as_posix(),
    "left_spring_link"  : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_spring_link.ply").as_posix(),
    "left_follower"     : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_follower.ply").as_posix(),

    "right_driver"      : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_driver.ply").as_posix(),
    "right_coupler"     : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_coupler.ply").as_posix(),
    "right_spring_link" : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_spring_link.ply").as_posix(),
    "right_follower"    : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_follower.ply").as_posix(),

    "background"        : (_ASSETS_FRANKA_DIR / "3dgs" / "background.ply").as_posix(),
    "banana"            : (_ASSETS_BANANA_DIR / "3dgs" / "fruit_banana.ply").as_posix()
}

mj_model = mujoco.MjModel.from_xml_path(mjcf_path.as_posix())
mj_data = mujoco.MjData(mj_model)
mujoco.mj_step(mj_model, mj_data)

gsmj_renderer = GSRendererMuJoCo(gaussians, mj_model)
gsmj_renderer.update_gaussians(mj_data)
results = gsmj_renderer.render(mj_model, mj_data, list(range(mj_model.ncam)), 320, 240)
print("MuJoCo render results:")
for k, (rgb, depth) in results.items():
    rgb_np = rgb.cpu().numpy()
    mediapy.show_image(rgb_np)

# Load the scene model
mx_model = load_model(mjcf_path.as_posix())
mx_data = SceneData(mx_model)

# Initialize the simulation
step(mx_model, mx_data)

gsmx_renderer = GSRendererMotrixSim(gaussians, mx_model)
gsmx_renderer.update_gaussians(mx_data)
results = gsmx_renderer.render(mx_model, mx_data, list(range(len(mx_model.cameras))), 320, 240)
print("MotrixSim render results:")
for k, (rgb, depth) in results.items():
    rgb_np = rgb.cpu().numpy()
    mediapy.show_image(rgb_np)

In [ ]:
print(mx_model.num_bodies, mj_model.nbody)
print(mx_model.num_links, mj_model.njnt)
print(mx_model.num_geoms, mj_model.ngeom)

print(mj_data.xpos.shape, mj_data.xquat.shape, mx_model.get_link_poses(mx_data).shape)
print(mj_data.geom_xpos.shape, mj_data.geom_xmat.shape)

assert np.allclose(mj_data.xpos[1:], mx_model.get_link_poses(mx_data)[:,:3])
assert np.allclose(mj_data.xmat[1:], Rotation.from_quat(mx_model.get_link_poses(mx_data)[:,3:]).as_matrix().reshape(-1,9), atol=1e-6)